# Tests de integración de `validate_trips()`

Notebook orientado a verificar el comportamiento end-to-end de OP-02 `validate trips`,
considerando resultado tabular, resultado operacional y trazabilidad.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "synthetic" 

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:
- imports generales,
- imports del módulo,
- helpers de testing reutilizables,
- y configuración básica de display.

### 0.1 Imports generales

In [4]:
import json
from pprint import pprint
from copy import deepcopy
import pandas as pd
import numpy as np

### 0.2 Imports del módulo 

In [5]:
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema, TripSchemaEffective
from pylondrina.datasets import TripDataset
from pylondrina.reports import ValidationReport
from pylondrina.errors import ValidationError
from pylondrina.validation import ValidationOptions, validate_trips
from pylondrina.importing import ImportOptions, import_trips_from_dataframe

### 0.3 Helpers de testing reutilizables

In [6]:
def show_ok(label: str):
    print(f"OK - {label}")

def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj, default=str)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e

def get_issue_codes(issues):
    return [i.code if hasattr(i, "code") else i.get("code") for i in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró el issue {code}. Codes actuales: {codes}"

def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente el issue {code}. Codes actuales: {codes}"

def assert_counts_by_level(issues, *, errors=None, warnings=None, info=None):
    counts = {"error": 0, "warning": 0, "info": 0}
    for issue in issues:
        counts[issue.level] = counts.get(issue.level, 0) + 1

    if errors is not None:
        assert counts["error"] == errors, f"errors esperado={errors}, actual={counts['error']}"
    if warnings is not None:
        assert counts["warning"] == warnings, f"warnings esperado={warnings}, actual={counts['warning']}"
    if info is not None:
        assert counts["info"] == info, f"info esperado={info}, actual={counts['info']}"

### 0.4 Configuración de display

In [7]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Sección 1. Integration tests de validate_trips

### Imports y fixtures reutilizables

Qué contiene: imports específicos para integración, schemas/fixtures base y helpers de clonación para no contaminar metadata entre tests.

In [8]:
def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


CANONICAL_MODE_VALUES = [
    "walk",
    "bicycle",
    "scooter",
    "motorcycle",
    "car",
    "taxi",
    "ride_hailing",
    "bus",
    "metro",
    "train",
    "other",
]

CANONICAL_PURPOSE_VALUES = [
    "home",
    "work",
    "education",
    "shopping",
    "errand",
    "health",
    "leisure",
    "transfer",
    "other",
]

PURPOSE_FINOS = {
    "Al trabajo",
    "Por trabajo",
    "Al estudio",
    "Por estudio",
    "volver a casa",
    "Visitar a alguien",
    "Buscar o Dejar a alguien",
    "Buscar o dejar algo",
    "Comer o Tomar algo",
    "De compras",
    "Trámites",
    "Recreación",
    "Otra actividad",
}


# ------------------------------------------------------------------
# Schema directo para fixtures de OP-02 (sin pasar por import)
# ------------------------------------------------------------------

schema_validate_direct = TripSchema(
    version="1.1",
    fields={
        "movement_id": make_field("movement_id", "string", required=True),
        "user_id": make_field("user_id", "string", required=True),
        "origin_latitude": make_field("origin_latitude", "float", required=True),
        "origin_longitude": make_field("origin_longitude", "float", required=True),
        "destination_latitude": make_field("destination_latitude", "float", required=True),
        "destination_longitude": make_field("destination_longitude", "float", required=True),
        "origin_time_utc": make_field("origin_time_utc", "datetime", required=False),
        "destination_time_utc": make_field("destination_time_utc", "datetime", required=False),
        "mode": make_field(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(values=CANONICAL_MODE_VALUES, extendable=False),
        ),
        "purpose": make_field(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(values=CANONICAL_PURPOSE_VALUES, extendable=True),
        ),
        "survey_wave": make_field(
            "survey_wave",
            "string",
            required=False,
            constraints={"nullable": False},
        ),
    },
    required=[
        "movement_id",
        "user_id",
        "origin_latitude",
        "origin_longitude",
        "destination_latitude",
        "destination_longitude",
    ],
    semantic_rules=None,
)

schema_effective_validate_direct = TripSchemaEffective(
    domains_effective={
        "mode": {
            "values": CANONICAL_MODE_VALUES,
            "extendable": False,
            "extended": False,
            "added_values": [],
            "unknown_value": "unknown",
        },
        "purpose": {
            "values": CANONICAL_PURPOSE_VALUES,
            "extendable": True,
            "extended": False,
            "added_values": [],
            "unknown_value": "unknown",
        },
    },
    temporal={"tier": "tier_1"},
    fields_effective=list(schema_validate_direct.fields.keys()),
)


def make_tripdataset_direct(
    df: pd.DataFrame,
    *,
    dataset_id: str,
    schema: TripSchema = schema_validate_direct,
    schema_effective: TripSchemaEffective = schema_effective_validate_direct,
    metadata: dict | None = None,
) -> TripDataset:
    base_metadata = {
        "dataset_id": dataset_id,
        # Intencionalmente dejamos sin "events" en algunas fixtures para probar
        # que validate_trips la cree cuando falte.
        "is_validated": False,
        "domains_effective": deepcopy(schema_effective.domains_effective),
        "temporal": {
            "tier": "tier_1",
            "fields_present": ["origin_time_utc", "destination_time_utc"],
        },
    }
    if metadata:
        base_metadata.update(deepcopy(metadata))

    return TripDataset(
        data=df.copy(),
        schema=schema,
        schema_version=schema.version,
        provenance={
            "source": {
                "name": "synthetic_direct",
                "entity": "trips",
                "version": "validate_op02_integration",
            }
        },
        field_correspondence={},
        value_correspondence={},
        metadata=base_metadata,
        schema_effective=deepcopy(schema_effective),
    )


def clone_tripdataset(trips: TripDataset) -> TripDataset:
    return TripDataset(
        data=trips.data.copy(deep=True),
        schema=trips.schema,
        schema_version=trips.schema_version,
        provenance=deepcopy(trips.provenance),
        field_correspondence=deepcopy(trips.field_correspondence),
        value_correspondence=deepcopy(trips.value_correspondence),
        metadata=deepcopy(trips.metadata),
        schema_effective=deepcopy(trips.schema_effective),
    )


# ------------------------------------------------------------------
# Fixtures directas de Capa 1
# ------------------------------------------------------------------

tripdataset_unvalidated_small = make_tripdataset_direct(
    pd.DataFrame({
        "movement_id": ["m1", "m2", "m3"],
        "user_id": ["u1", "u2", "u3"],
        "origin_latitude": [-33.45, -33.46, -33.47],
        "origin_longitude": [-70.66, -70.67, -70.68],
        "destination_latitude": [-33.43, -33.44, -33.45],
        "destination_longitude": [-70.61, -70.62, -70.63],
        "origin_time_utc": [
            "2026-04-02T08:00:00Z",
            "2026-04-02T09:00:00Z",
            "2026-04-02T10:00:00Z",
        ],
        "destination_time_utc": [
            "2026-04-02T08:30:00Z",
            "2026-04-02T09:20:00Z",
            "2026-04-02T10:25:00Z",
        ],
        "mode": ["walk", "bus", "metro"],
        "purpose": ["work", "education", "health"],
        "survey_wave": ["w1", "w1", "w1"],
    }),
    dataset_id="tds-unvalidated-small",
    metadata={"events": []},
)

tripdataset_with_domain_issues = make_tripdataset_direct(
    pd.DataFrame({
        "movement_id": ["m1", "m2", "m3", "m4"],
        "user_id": ["u1", "u2", "u3", "u4"],
        "origin_latitude": [-33.45, -33.46, -33.47, -33.48],
        "origin_longitude": [-70.66, -70.67, -70.68, -70.69],
        "destination_latitude": [-33.43, -33.44, -33.45, -33.46],
        "destination_longitude": [-70.61, -70.62, -70.63, -70.64],
        "origin_time_utc": [
            "2026-04-02T08:00:00Z",
            "2026-04-02T09:00:00Z",
            "2026-04-02T10:00:00Z",
            "2026-04-02T11:00:00Z",
        ],
        "destination_time_utc": [
            "2026-04-02T08:30:00Z",
            "2026-04-02T09:30:00Z",
            "2026-04-02T10:30:00Z",
            "2026-04-02T11:30:00Z",
        ],
        "mode": ["walk", "teleport", "bus", "metro"],
        "purpose": ["work", "Otra actividad", "education", "health"],
        "survey_wave": ["w1", "w1", "w1", "w1"],
    }),
    dataset_id="tds-domain-issues",
    metadata={"events": []},
)

tripdataset_with_temporal_issues = make_tripdataset_direct(
    pd.DataFrame({
        "movement_id": ["m1", "m2"],
        "user_id": ["u1", "u2"],
        "origin_latitude": [-33.45, -33.46],
        "origin_longitude": [-70.66, -70.67],
        "destination_latitude": [-33.43, -33.44],
        "destination_longitude": [-70.61, -70.62],
        "origin_time_utc": [
            "2026-04-02T09:00:00Z",
            "2026-04-02T10:00:00Z",
        ],
        "destination_time_utc": [
            "2026-04-02T08:30:00Z",
            "2026-04-02T09:50:00Z",
        ],
        "mode": ["walk", "bus"],
        "purpose": ["work", "education"],
        "survey_wave": ["w1", "w1"],
    }),
    dataset_id="tds-temporal-issues",
    metadata={"events": []},
)


# ------------------------------------------------------------------
# Schema y opciones para Capa 2 (puente import -> validate)
# ------------------------------------------------------------------

schema_trips_canonical = TripSchema(
    version="1.1",
    fields={
        "movement_id": make_field("movement_id", "string", required=True),
        "user_id": make_field("user_id", "string", required=True),
        "origin_longitude": make_field("origin_longitude", "float", required=True),
        "origin_latitude": make_field("origin_latitude", "float", required=True),
        "destination_longitude": make_field("destination_longitude", "float", required=True),
        "destination_latitude": make_field("destination_latitude", "float", required=True),
        "origin_h3_index": make_field("origin_h3_index", "string", required=True),
        "destination_h3_index": make_field("destination_h3_index", "string", required=True),
        "trip_id": make_field("trip_id", "string", required=True),
        "movement_seq": make_field("movement_seq", "int", required=True),
        "origin_time_utc": make_field("origin_time_utc", "datetime", required=False),
        "destination_time_utc": make_field("destination_time_utc", "datetime", required=False),
        "mode": make_field(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(values=CANONICAL_MODE_VALUES, extendable=False),
        ),
        "purpose": make_field(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(values=CANONICAL_PURPOSE_VALUES, extendable=True),
        ),
        "user_gender": make_field(
            "user_gender",
            "categorical",
            required=False,
            domain=DomainSpec(values=["female", "male", "other", "unknown"], extendable=False),
        ),
        "origin_municipality": make_field("origin_municipality", "string", required=False),
        "destination_municipality": make_field("destination_municipality", "string", required=False),
        "trip_weight": make_field("trip_weight", "float", required=False),
        "timezone_offset_min": make_field("timezone_offset_min", "int", required=False),
    },
    required=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
    ],
    semantic_rules=None,
)

import_options_bridge = ImportOptions(
    keep_extra_fields=True,
    strict=False,
    strict_domains=False,
    single_stage=False,
)

validation_options_bridge = ValidationOptions(
    strict=False,
    validate_required_fields=True,
    validate_types_and_formats=True,
    validate_constraints=True,
    validate_domains="full",
    validate_temporal_consistency=True,
    validate_duplicates=True,
    duplicates_subset=("user_id", "origin_time_utc", "origin_h3_index", "destination_h3_index"),
    allow_partial_od_spatial=True,
)

### Test 1 - caso principal correcto: dataset pequeño no validado

Qué prueba: el camino principal correcto de OP-02 sobre una fixture directa, incluyendo ValidationReport, event, issues_summary, summary, is_validated y no mutación de data. Este es el caso base para decir que la operación funciona bien en un dataset normal no validado todavía

In [12]:
trips = clone_tripdataset(tripdataset_unvalidated_small)
data_before = trips.data.copy(deep=True)

report = validate_trips(
    trips,
    options=ValidationOptions(
        validate_domains="full",
        validate_temporal_consistency=True,
        validate_duplicates=False,
        allow_partial_od_spatial=True,
    ),
)

assert isinstance(report, ValidationReport)
assert report.ok is True
assert len(report.issues) == 0

assert trips.metadata["is_validated"] is True
assert "events" in trips.metadata
assert len(trips.metadata["events"]) == 1

event = trips.metadata["events"][-1]
assert event["op"] == "validate_trips"
assert "ts_utc" in event
assert "parameters" in event
assert "summary" in event
assert "issues_summary" in event
assert event["summary"] == report.summary

assert report.summary["ok"] is True
assert report.summary["n_rows"] == len(trips.data)
assert report.summary["n_errors"] == 0
assert report.summary["n_warnings"] == 0
assert report.summary["checks_executed"]["required_fields"] is True
assert report.summary["checks_executed"]["constraints"] is True
assert report.summary["checks_executed"]["types_and_formats"] is True
assert report.summary["checks_executed"]["domains"] is True
assert report.summary["checks_executed"]["temporal_consistency"] is True

pd.testing.assert_frame_equal(trips.data, data_before)

assert_json_safe(report.summary, "report.summary")
assert_json_safe(event, "validate_event")


display(trips.data)
display(report.issues)
show_ok("Test 1 - caso principal correcto")

,movement_id,user_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,origin_time_utc,destination_time_utc,mode,purpose,survey_wave
0,m1,u1,-33.45,-70.66,-33.43,-70.61,2026-04-02T08:00:00Z,2026-04-02T08:30:00Z,walk,work,w1
1,m2,u2,-33.46,-70.67,-33.44,-70.62,2026-04-02T09:00:00Z,2026-04-02T09:20:00Z,bus,education,w1
2,m3,u3,-33.47,-70.68,-33.45,-70.63,2026-04-02T10:00:00Z,2026-04-02T10:25:00Z,metro,health,w1


[]

OK - Test 1 - caso principal correcto


### Test 2 - required column missing sobre fixture directa

Qué prueba: un error de conformidad por datos/estructura observable, no fatal de config. Aquí validamos que falte una columna requerida y que eso se traduzca en issue, ok=False, evento y is_validated=False, sin excepción si strict=False. Esto cubre required fields dentro de la integración real de la función pública.

In [15]:
trips = clone_tripdataset(tripdataset_unvalidated_small)
trips.data = trips.data.drop(columns=["user_id"])

report = validate_trips(
    trips,
    options=ValidationOptions(
        strict=False,
        validate_required_fields=True,
        validate_constraints=False,
        validate_types_and_formats=False,
        validate_domains="off",
        validate_temporal_consistency=False,
        validate_duplicates=False,
    ),
)

assert report.ok is False
assert_issue_present(report.issues, "VAL.CORE.REQUIRED_COLUMNS_MISSING")
assert_counts_by_level(report.issues, errors=1, warnings=0, info=0)

assert trips.metadata["is_validated"] is False
assert len(trips.metadata["events"]) == 1
assert trips.metadata["events"][-1]["op"] == "validate_trips"

display(report.issues)
display(report.summary)
show_ok("Test 2 - required column missing")

[Issue(level='error', code='VAL.CORE.REQUIRED_COLUMNS_MISSING', message="Faltan columnas requeridas según el TripSchema: ['user_id'].", field=None, source_field=None, row_count=1, details={'check': 'required_columns', 'missing_required': ['user_id'], 'required': ['movement_id', 'user_id'], 'available_columns_sample': ['movement_id', 'origin_latitude', 'origin_longitude', 'destination_latitude', 'destination_longitude', 'origin_time_utc', 'destination_time_utc', 'mode', 'purpose', 'survey_wave'], 'available_columns_total': 10, 'action': 'report_error'})]

{'ok': False,
 'n_rows': 3,
 'n_issues': 1,
 'n_errors': 1,
 'n_warnings': 0,
 'n_info': 0,
 'counts_by_level': {'error': 1, 'warning': 0, 'info': 0},
 'counts_by_code': {'VAL.CORE.REQUIRED_COLUMNS_MISSING': 1},
 'checked_fields': ['movement_id'],
 'checks_executed': {'required_fields': True,
  'types_and_formats': False,
  'constraints': False,
  'domains': False,
  'temporal_consistency': False,
  'duplicates': False},
 'schema_version': '1.1',
 'limits': {'max_issues': 500,
  'issues_truncated': False,
  'n_issues_emitted': 1,
  'n_issues_detected_total': 1}}

OK - Test 2 - required column missing


### Test 3 - nullabilidad efectiva sobre fixture directa

Qué prueba: una violación de nullable efectivo integrada en la función pública. Usamos survey_wave, que no es required pero sí tiene nullable=False, para verificar que validate detecta correctamente la nullabilidad efectiva y no solo required fields.

In [18]:
trips = clone_tripdataset(tripdataset_unvalidated_small)
trips.data.loc[1, "survey_wave"] = None

report = validate_trips(
    trips,
    options=ValidationOptions(
        strict=False,
        validate_required_fields=True,
        validate_constraints=True,
        validate_types_and_formats=False,
        validate_domains="off",
        validate_temporal_consistency=False,
        validate_duplicates=False,
    ),
)

assert report.ok is False
assert_issue_present(report.issues, "VAL.CORE.NULLABILITY_VIOLATION")
assert_counts_by_level(report.issues, errors=1, warnings=0, info=0)

assert trips.metadata["is_validated"] is False
assert len(trips.metadata["events"]) == 1

display(report.issues)
show_ok("Test 3 - nullabilidad efectiva")

[Issue(level='error', code='VAL.CORE.NULLABILITY_VIOLATION', message="El campo 'survey_wave' no admite nulos según nullable efectivo, pero se detectaron filas con valores faltantes.", field='survey_wave', source_field=None, row_count=1, details={'check': 'constraints', 'n_rows_total': 3, 'n_violations': 1, 'row_indices_sample': [1], 'values_sample': [None], 'raw_values_sample': [None], 'field': 'survey_wave', 'nullable_effective': False, 'action': 'report_error'})]

OK - Test 3 - nullabilidad efectiva


### Test 4 - caso degradado con warning de dominios

Qué prueba: un caso degradado/no fatal con warning, usando la fixture con issues de dominio. Aquí queremos que haya valores fuera de dominio pero que el ratio cumpla el mínimo, para obtener PARTIAL_COVERAGE, ok=True, evento y is_validated=True. Este es el caso “degradado o con warning” de la batería mínima.

In [20]:
trips = clone_tripdataset(tripdataset_with_domain_issues)

report = validate_trips(
    trips,
    options=ValidationOptions(
        strict=False,
        validate_required_fields=True,
        validate_constraints=True,
        validate_types_and_formats=True,
        validate_domains="full",
        domains_min_in_domain_ratio=0.50,
        validate_temporal_consistency=False,
        validate_duplicates=False,
    ),
)

assert report.ok is True
assert_issue_present(report.issues, "VAL.DOMAIN.PARTIAL_COVERAGE")
assert_counts_by_level(report.issues, errors=0, warnings=2, info=0)

assert "domains" in report.summary
assert report.summary["domains"]["mode"] == "full"
assert report.summary["domains"]["min_required_ratio"] == 0.50

assert trips.metadata["is_validated"] is True
assert len(trips.metadata["events"]) == 1
assert trips.metadata["events"][-1]["op"] == "validate_trips"

display(report.issues)
show_ok("Test 4 - caso degradado con warning de dominios")

[Issue(level='warning', code='VAL.DOMAIN.PARTIAL_COVERAGE', message="El campo 'mode' tiene cobertura parcial en dominio (0.7500); supera el mínimo, pero no alcanza cobertura completa.", field='mode', source_field=None, row_count=1, details={'check': 'domains', 'n_rows_total': 4, 'n_violations': 1, 'row_indices_sample': [1], 'values_sample': ['teleport'], 'raw_values_sample': ['teleport'], 'field': 'mode', 'mode': 'full', 'ratio_in_domain': 0.75, 'min_required_ratio': 0.5, 'n_checked_non_null': 4, 'n_in_domain': 3, 'domain_values_sample': ['bicycle', 'bus', 'car', 'metro', 'motorcycle', 'other', 'ride_hailing', 'scooter', 'taxi', 'train'], 'action': 'report_warning'}),
 Issue(level='warning', code='VAL.DOMAIN.PARTIAL_COVERAGE', message="El campo 'purpose' tiene cobertura parcial en dominio (0.7500); supera el mínimo, pero no alcanza cobertura completa.", field='purpose', source_field=None, row_count=1, details={'check': 'domains', 'n_rows_total': 4, 'n_violations': 1, 'row_indices_sampl

OK - Test 4 - caso degradado con warning de dominios


### Test 5 - temporal inconsistency con strict=False

Qué prueba: error temporal integrado, pero sin política strict. Debe retornar ValidationReport, dejar evento y marcar is_validated=False. Este es el caso base para temporal inconsistency como error de datos.

In [21]:
trips = clone_tripdataset(tripdataset_with_temporal_issues)
data_before = trips.data.copy(deep=True)

report = validate_trips(
    trips,
    options=ValidationOptions(
        strict=False,
        validate_required_fields=True,
        validate_constraints=True,
        validate_types_and_formats=True,
        validate_domains="off",
        validate_temporal_consistency=True,
        validate_duplicates=False,
    ),
)

assert report.ok is False
assert_issue_present(report.issues, "VAL.TEMPORAL.ORIGIN_AFTER_DESTINATION")
assert_counts_by_level(report.issues, errors=1, warnings=0, info=0)

assert "temporal" in report.summary
assert report.summary["temporal"]["evaluated"] is True
assert report.summary["temporal"]["n_violations"] == 2

assert trips.metadata["is_validated"] is False
assert len(trips.metadata["events"]) == 1
assert trips.metadata["events"][-1]["op"] == "validate_trips"

pd.testing.assert_frame_equal(trips.data, data_before)

display(report.issues)
show_ok("Test 5 - temporal inconsistency con strict=False")

[Issue(level='error', code='VAL.TEMPORAL.ORIGIN_AFTER_DESTINATION', message='Se detectaron filas donde origin_time_utc es posterior a destination_time_utc.', field=None, source_field=None, row_count=2, details={'check': 'temporal_consistency', 'n_rows_total': 2, 'n_violations': 2, 'row_indices_sample': [0, 1], 'values_sample': [{'origin_time_utc': '2026-04-02T09:00:00Z', 'destination_time_utc': '2026-04-02T08:30:00Z'}, {'origin_time_utc': '2026-04-02T10:00:00Z', 'destination_time_utc': '2026-04-02T09:50:00Z'}], 'raw_values_sample': [{'origin_time_utc': '2026-04-02T09:00:00Z', 'destination_time_utc': '2026-04-02T08:30:00Z'}, {'origin_time_utc': '2026-04-02T10:00:00Z', 'destination_time_utc': '2026-04-02T09:50:00Z'}], 'origin_field': 'origin_time_utc', 'destination_field': 'destination_time_utc', 'action': 'report_error'})]

OK - Test 5 - temporal inconsistency con strict=False


### Test 6 - política clave: strict=True con error temporal

Qué prueba: la política central de OP-02: con strict=True y error de datos, primero se registra la evidencia y recién después se lanza. Deben quedar event e is_validated=False aunque se levante excepción.

In [26]:
trips = clone_tripdataset(tripdataset_with_temporal_issues)

raised = None
try:
    validate_trips(
        trips,
        options=ValidationOptions(
            strict=True,
            validate_required_fields=True,
            validate_constraints=True,
            validate_types_and_formats=True,
            validate_domains="off",
            validate_temporal_consistency=True,
            validate_duplicates=False,
        ),
    )
except ValidationError as e:
    raised = e

assert raised is not None
assert isinstance(raised, ValidationError)

assert trips.metadata["is_validated"] is False
assert len(trips.metadata["events"]) == 1

event = trips.metadata["events"][-1]
assert event["op"] == "validate_trips"
assert "issues_summary" in event
assert event["summary"]["ok"] is False

display(raised)
display(raised.issues)
show_ok("Test 6 - strict=True con error temporal")

ValidationError(message='validate_trips detectó errores de datos y strict=True exige abortar.', code='VAL.TEMPORAL.ORIGIN_AFTER_DESTINATION', details={'check': 'temporal_consistency', 'n_rows_total': 2, 'n_violations': 2, 'row_indices_sample': [0, 1], 'values_sample': [{'origin_time_utc': '2026-04-02T09:00:00Z', 'destination_time_utc': '2026-04-02T08:30:00Z'}, {'origin_time_utc': '2026-04-02T10:00:00Z', 'destination_time_utc': '2026-04-02T09:50:00Z'}], 'raw_values_sample': [{'origin_time_utc': '2026-04-02T09:00:00Z', 'destination_time_utc': '2026-04-02T08:30:00Z'}, {'origin_time_utc': '2026-04-02T10:00:00Z', 'destination_time_utc': '2026-04-02T09:50:00Z'}], 'origin_field': 'origin_time_utc', 'destination_field': 'destination_time_utc', 'action': 'report_error'}, issue=Issue(level='error', code='VAL.TEMPORAL.ORIGIN_AFTER_DESTINATION', message='Se detectaron filas donde origin_time_utc es posterior a destination_time_utc.', field=None, source_field=None, row_count=2, details={'check': 't

(Issue(level='error', code='VAL.TEMPORAL.ORIGIN_AFTER_DESTINATION', message='Se detectaron filas donde origin_time_utc es posterior a destination_time_utc.', field=None, source_field=None, row_count=2, details={'check': 'temporal_consistency', 'n_rows_total': 2, 'n_violations': 2, 'row_indices_sample': [0, 1], 'values_sample': [{'origin_time_utc': '2026-04-02T09:00:00Z', 'destination_time_utc': '2026-04-02T08:30:00Z'}, {'origin_time_utc': '2026-04-02T10:00:00Z', 'destination_time_utc': '2026-04-02T09:50:00Z'}], 'raw_values_sample': [{'origin_time_utc': '2026-04-02T09:00:00Z', 'destination_time_utc': '2026-04-02T08:30:00Z'}, {'origin_time_utc': '2026-04-02T10:00:00Z', 'destination_time_utc': '2026-04-02T09:50:00Z'}], 'origin_field': 'origin_time_utc', 'destination_field': 'destination_time_utc', 'action': 'report_error'}),)

OK - Test 6 - strict=True con error temporal


### Test 7 - fatal/precondición: duplicates subset no usable

Qué prueba: un fatal de config/precondición. Aquí no debe existir ValidationReport ni evento nuevo, porque la operación aborta antes del pipeline normal. Este es el caso fatal/precondición de la batería mínima

In [27]:
trips = clone_tripdataset(tripdataset_unvalidated_small)
events_before = deepcopy(trips.metadata.get("events", []))
validated_before = trips.metadata.get("is_validated", False)

raised = None
try:
    validate_trips(
        trips,
        options=ValidationOptions(
            validate_duplicates=True,
            duplicates_subset=None,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ValidationError)

assert trips.metadata.get("events", []) == events_before
assert trips.metadata.get("is_validated", False) == validated_before

display(raised.issues)
show_ok("Test 7 - fatal de config por duplicates_subset no usable")

(Issue(level='error', code='VAL.CONFIG.DUPLICATES_SUBSET_NOT_PROVIDED', message='validate_duplicates=True requiere duplicates_subset explícito; no se definió subset usable.', field=None, source_field=None, row_count=None, details={'validate_duplicates': True, 'duplicates_subset': None, 'action': 'abort'}),)

OK - Test 7 - fatal de config por duplicates_subset no usable


### Test 8 - metadata / event / summary consistente

Qué prueba: la consistencia del contrato observable: summary del reporte debe ser exactamente el mismo que se guarda en el evento, y issues_summary debe ser JSON-safe. Este test es redundante con el happy path, pero útil como chequeo explícito de trazabilidad.

In [28]:
trips = clone_tripdataset(tripdataset_with_domain_issues)

report = validate_trips(
    trips,
    options=ValidationOptions(
        validate_domains="full",
        domains_min_in_domain_ratio=0.50,
        validate_temporal_consistency=False,
        validate_duplicates=False,
    ),
)

event = trips.metadata["events"][-1]

assert event["summary"] == report.summary
assert "issues_summary" in event
assert_json_safe(event["issues_summary"], "event.issues_summary")
assert_json_safe(event, "validate_event")
assert_json_safe(report.summary, "validation_report_summary")

display(report.issues)
show_ok("Test 8 - metadata / event / summary consistente")

[Issue(level='warning', code='VAL.DOMAIN.PARTIAL_COVERAGE', message="El campo 'mode' tiene cobertura parcial en dominio (0.7500); supera el mínimo, pero no alcanza cobertura completa.", field='mode', source_field=None, row_count=1, details={'check': 'domains', 'n_rows_total': 4, 'n_violations': 1, 'row_indices_sample': [1], 'values_sample': ['teleport'], 'raw_values_sample': ['teleport'], 'field': 'mode', 'mode': 'full', 'ratio_in_domain': 0.75, 'min_required_ratio': 0.5, 'n_checked_non_null': 4, 'n_in_domain': 3, 'domain_values_sample': ['bicycle', 'bus', 'car', 'metro', 'motorcycle', 'other', 'ride_hailing', 'scooter', 'taxi', 'train'], 'action': 'report_warning'}),
 Issue(level='warning', code='VAL.DOMAIN.PARTIAL_COVERAGE', message="El campo 'purpose' tiene cobertura parcial en dominio (0.7500); supera el mínimo, pero no alcanza cobertura completa.", field='purpose', source_field=None, row_count=1, details={'check': 'domains', 'n_rows_total': 4, 'n_violations': 1, 'row_indices_sampl

OK - Test 8 - metadata / event / summary consistente


## Seccion 3: Test puente con Import

### Test 9 - puente real import -> validate con tripdataset_real_canonical

Qué prueba: el handoff real entre operaciones. No re-prueba OP-01 completo; comprueba que import deja el contexto que validate necesita y que validate lo consume correctamente:

- schema_effective,
- domains_effective,
- metadata["temporal"],
- metadata["h3"],
- eventos import_trips y validate_trips,
- y is_validated=True al final.

In [34]:
csv_path = DATA_PATH / "tripdataset_real_canonical.csv"
df_real_canonical = pd.read_csv(csv_path)

tripdataset_bridge, import_report = import_trips_from_dataframe(
    df_real_canonical,
    schema=schema_trips_canonical,
    source_name="tripdataset_real_canonical",
    options=import_options_bridge,
    provenance={
        "source": {
            "name": "tripdataset_real_canonical",
            "entity": "trips",
            "version": "synthetic_v1",
        }
    },
    h3_resolution=8,
)

# -----------------------------
# Verificaciones de handoff tras import
# -----------------------------
assert tripdataset_bridge.schema_effective is not None
assert "domains_effective" in tripdataset_bridge.metadata
assert "temporal" in tripdataset_bridge.metadata
assert "h3" in tripdataset_bridge.metadata
assert tripdataset_bridge.metadata["temporal"]["tier"] == "tier_1"
assert tripdataset_bridge.metadata["h3"]["resolution"] == 8
assert len(tripdataset_bridge.metadata["events"]) >= 1
assert tripdataset_bridge.metadata["events"][0]["op"] == "import_trips"

# Se guarda snapshot del dataframe antes de validate para asegurar no mutación.
data_before_validate = tripdataset_bridge.data.copy(deep=True)

validation_report = validate_trips(
    tripdataset_bridge,
    options=validation_options_bridge,
)

# -----------------------------
# Verificaciones del validate
# -----------------------------
assert isinstance(validation_report, ValidationReport)
assert validation_report.ok is True
assert not any(issue.level == "error" for issue in validation_report.issues)

assert tripdataset_bridge.metadata["is_validated"] is True
assert len(tripdataset_bridge.metadata["events"]) == 2
assert tripdataset_bridge.metadata["events"][0]["op"] == "import_trips"
assert tripdataset_bridge.metadata["events"][1]["op"] == "validate_trips"

validate_event = tripdataset_bridge.metadata["events"][-1]
assert validate_event["summary"] == validation_report.summary
assert "issues_summary" in validate_event

assert "domains" in validation_report.summary
assert "temporal" in validation_report.summary
assert "duplicates" in validation_report.summary

assert validation_report.summary["temporal"]["evaluated"] is True
assert validation_report.summary["duplicates"]["evaluated"] is True

pd.testing.assert_frame_equal(tripdataset_bridge.data, data_before_validate)

display(tripdataset_bridge.data.head())
display(import_report.issues)

display(validation_report.issues)
display(tripdataset_bridge.metadata["events"])

show_ok("Test 9 - puente real import -> validate")

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,mode,purpose,user_gender,origin_municipality,destination_municipality,trip_weight,timezone_offset_min,personal_income_band,origin_stop_id,origin_sector_id,household_size,origin_h3_index,destination_h3_index
0,m00000,u0101,t00000,0,-70.650462,-33.522064,-70.620669,-33.559297,2026-03-07 13:56:00+00:00,2026-03-07 14:11:00+00:00,bicycle,Por estudio,other,Quilicura,Pudahuel,0.908,-180,200k-400k,STOP_3352,SEC_253,1,88b2c5461dfffff,88b2c54681fffff
1,m00001,u0140,t00001,0,-70.845876,-33.389812,-70.864414,-33.393808,2026-03-01 20:05:00+00:00,2026-03-01 21:52:00+00:00,car,De compras,female,Pudahuel,Quilicura,2.263,-180,200k-400k,STOP_2840,SEC_276,4,88b2c551c1fffff,88b2c551e9fffff
2,m00002,u0487,t00002,0,-70.773453,-33.422863,-70.754738,-33.446732,2026-03-06 16:34:00+00:00,2026-03-06 17:59:00+00:00,train,Comer o Tomar algo,female,Vitacura,San Miguel,1.674,-180,400k-800k,STOP_3679,SEC_232,4,88b2c550bbfffff,88b2c5552bfffff
3,m00003,u0252,t00003,0,-70.720751,-33.473752,-70.689224,-33.474590,2026-03-07 22:47:00+00:00,2026-03-08 00:32:00+00:00,other,Por trabajo,unknown,Independencia,Recoleta,3.562,-180,<200k,STOP_1308,SEC_524,2,88b2c555e3fffff,88b2c555d5fffff
4,m00004,u0122,t00004,0,-70.831438,-33.474356,-70.854588,-33.409434,2026-03-04 07:35:00+00:00,2026-03-04 09:09:00+00:00,motorcycle,Al estudio,unknown,Providencia,San Miguel,1.166,-180,no_income,STOP_3178,SEC_027,6,88b2c54049fffff,88b2c5518dfffff


[Issue(level='info', code='DOM.EXTENSION.APPLIED', message="Se extendió el dominio de 'purpose' con 13 valores nuevos.", field='purpose', source_field=None, row_count=None, details={'field': 'purpose', 'n_added': 13, 'added_values_sample': ['Al estudio', 'Al trabajo', 'Buscar o Dejar a alguien', 'Buscar o dejar algo', 'Comer o Tomar algo'], 'added_values_total': 13, 'policy': 'extendable_non_strict', 'action': 'extended_domain'}),
 Issue(level='info', code='DOM.EXTENSION.APPLIED', message="Se extendió el dominio de 'purpose' con 13 valores nuevos.", field='purpose', source_field=None, row_count=None, details={'field': 'purpose', 'n_added': 13, 'added_values_sample': ['Al estudio', 'Al trabajo', 'Buscar o Dejar a alguien', 'Buscar o dejar algo', 'Comer o Tomar algo'], 'added_values_total': 13, 'policy': 'extendable_non_strict', 'action': 'extended_domain'}),
 Issue(level='info', code='IMP.METADATA.DATASET_ID_CREATED', message="Se generó dataset_id para el dataset importado: 'tripds_1ae9

[]

[{'op': 'import_trips',
  'ts_utc': '2026-04-02T12:27:37.714140+00:00',
  'parameters': {'keep_extra_fields': True,
   'selected_fields': None,
   'strict': False,
   'strict_domains': False,
   'single_stage': False,
   'h3_resolution': 8,
   'source_name': 'tripdataset_real_canonical',
   'source_timezone': None},
  'summary': {'input_rows': 1200,
   'output_rows': 1200,
   'rows_dropped': 0,
   'n_fields_mapped': 0,
   'n_domain_mappings_applied': 0,
   'columns_added': ['origin_h3_index', 'destination_h3_index'],
   'columns_deleted': [],
   'domains_extended_count': 1,
   'temporal_tier': 'tier_1',
   'temporal_notes': None},
  'issues_summary': {'counts': {'info': 2},
   'by_code': {'DOM.EXTENSION.APPLIED': 2}}},
 {'op': 'validate_trips',
  'ts_utc': '2026-04-02T12:27:37Z',
  'parameters': {'strict': False,
   'max_issues': 500,
   'sample_rows_per_issue': 5,
   'validate_required_fields': True,
   'validate_types_and_formats': True,
   'validate_constraints': True,
   'validate_

OK - Test 9 - puente real import -> validate


### Test 10 - puente opcional: validate usa dominios efectivos de purpose

Qué prueba: solo si el import realmente extendió domains_effective["purpose"] con valores del grupo finos, este test confirma que validate_trips usa el dominio efectivo y no solo el dominio base del schema. Si no hay extensión real en el dataset importado, el test se deja como skip explícito.

In [36]:
tripdataset_bridge, _ = import_trips_from_dataframe(
    df_real_canonical,
    schema=schema_trips_canonical,
    source_name="tripdataset_real_canonical",
    options=import_options_bridge,
    provenance={
        "source": {
            "name": "tripdataset_real_canonical",
            "entity": "trips",
            "version": "synthetic_v1",
        }
    },
    h3_resolution=8,
)

purpose_effective = set(
    tripdataset_bridge.schema_effective.domains_effective.get("purpose", {}).get("values", [])
)

has_finos_extension = len(purpose_effective.intersection(PURPOSE_FINOS)) > 0

if not has_finos_extension:
    print("SKIP - Test 10: el dataset importado no quedó con extensión efectiva en purpose.")
else:
    report = validate_trips(
        tripdataset_bridge,
        options=ValidationOptions(
            strict=False,
            validate_required_fields=True,
            validate_types_and_formats=True,
            validate_constraints=True,
            validate_domains="full",
            validate_temporal_consistency=True,
            validate_duplicates=False,
            allow_partial_od_spatial=True,
        ),
    )

    assert_issue_absent(report.issues, "VAL.DOMAIN.RATIO_BELOW_MIN")
    assert tripdataset_bridge.metadata["is_validated"] is True
    assert tripdataset_bridge.metadata["events"][-1]["op"] == "validate_trips"

    display(report.issues)
    show_ok("Test 10 - validate usa dominios efectivos de purpose")

[]

OK - Test 10 - validate usa dominios efectivos de purpose


In [37]:
show_ok("Sección 3 completa - Integration tests de validate_trips")

OK - Sección 3 completa - Integration tests de validate_trips


## Resumen de cobertura que te queda con esta batería

Con estos tests se cubre el núcleo suficiente de OP-02 :

- caso principal correcto → Grupo 3.1
- fatal/precondición → Grupo 3.7
- caso degradado o con warning → Grupo 3.4
- metadata/evento/summary → Grupos 3.1, 3.8, 3.9
- política clave (strict) → Grupo 3.6
- required / nullability efectiva → Grupos 3.2 y 3.3
- domains → Grupo 3.4 y opcional 3.10
- temporal → Grupos 3.5 y 3.6
- handoff real import -> validate → Grupo 3.9
